# 🧮 Naive Bayes
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Naive Bayes extends the Bayesian classification idea with one strong simplification: assume features are conditionally independent given the class. Despite this "naive" assumption often being wrong, it works surprisingly well in practice — especially for text.

### What you'll learn
- The conditional independence assumption and why it's "naive"
- Gaussian Naive Bayes for numeric features
- Multinomial Naive Bayes for text classification
- Why NB works well even when the independence assumption is violated
- NB vs logistic regression — when to use each

### Dataset: SMS Spam (text classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

In [ ]:
# Load SMS Spam dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
try:
    sms = pd.read_csv(url, sep='\t', header=None, names=['label','message'])
    sms['spam'] = (sms['label']=='spam').astype(int)
    print(f"SMS Spam dataset: {len(sms):,} messages")
    print(f"Spam rate: {sms['spam'].mean():.1%}")
    print("\nSample messages:")
    print(sms[['label','message']].head(5).to_string(index=False))
except:
    # Fallback synthetic data
    from sklearn.datasets import fetch_20newsgroups
    news = fetch_20newsgroups(subset='train', categories=['sci.med','sci.space'], remove=('headers','footers'))
    sms = pd.DataFrame({'message': news.data, 'spam': news.target, 'label': ['med' if t==0 else 'space' for t in news.target]})
    print("Using 20 Newsgroups dataset (sci.med vs sci.space)")

## 📐 Part 1 — Bayes with Independence Assumption

Full Bayes would require estimating P(X₁=x₁, X₂=x₂, ..., Xₚ=xₚ | Y=k) — the joint distribution of all features. With p=1000 features this is impossible.

**Naive Bayes** assumes conditional independence:
```
P(X₁,...,Xₚ | Y=k) = P(X₁|Y=k) × P(X₂|Y=k) × ... × P(Xₚ|Y=k)
```

Now we only need to estimate p separate univariate distributions — completely tractable.

The "naive" part: features are rarely truly independent. But the resulting classifier is surprisingly robust — the probability estimates may be wrong, but the *decision* (which class gets the highest probability) is often right.

In [ ]:
# Gaussian NB — numeric features (Iris)
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target

gnb = GaussianNB()
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)
gnb.fit(X_tr, y_tr)

print("=== Gaussian Naive Bayes: Iris ===")
print(classification_report(y_te, gnb.predict(X_te), target_names=iris.target_names))

# Show what GNB learns: Gaussian distributions per class per feature
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()
colors = ['#1e5fa8','#e85d20','#1a7a45']

for ax, (feat_idx, feat) in enumerate(zip(range(4), iris.feature_names)):
    x_range = np.linspace(X_iris.iloc[:,feat_idx].min()-0.5, X_iris.iloc[:,feat_idx].max()+0.5, 200)
    from scipy.stats import norm
    for cls_idx, (color, name) in enumerate(zip(colors, iris.target_names)):
        mu  = gnb.theta_[cls_idx, feat_idx]
        std = np.sqrt(gnb.var_[cls_idx, feat_idx])
        ax.plot(x_range, norm.pdf(x_range, mu, std), color=color, lw=2, label=name)
        ax.hist(X_iris.iloc[y_iris==cls_idx, feat_idx], bins=15, 
                color=color, alpha=0.2, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Gaussian NB: Learned Distributions per Class per Feature', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 GNB fits a Gaussian to each feature within each class.")
print("   Classification = find which class's Gaussians best explain the observed values.")

In [ ]:
# Text classification: spam detection
X_text = sms['message'].values
y_text = sms['spam'].values

# Pipeline: TF-IDF → Multinomial NB
pipeline_mnb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf',   MultinomialNB(alpha=1.0))  # alpha = Laplace smoothing
])

X_tr, X_te, y_tr, y_te = train_test_split(X_text, y_text, test_size=0.2, 
                                             random_state=42, stratify=y_text)
pipeline_mnb.fit(X_tr, y_tr)
y_pred = pipeline_mnb.predict(X_te)

print("=== Multinomial Naive Bayes: Spam Detection ===\n")
print(classification_report(y_te, y_pred, target_names=['Ham','Spam']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Ham','Spam']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Spam Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Most discriminative words for spam vs ham
import re
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_vec = vectorizer.fit_transform(X_text)
mnb = MultinomialNB().fit(X_vec, y_text)

feature_names = vectorizer.get_feature_names_out()
# Log-ratio: how much more likely in spam vs ham
log_ratios = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]

top_spam = pd.Series(log_ratios, index=feature_names).nlargest(15)
top_ham  = pd.Series(log_ratios, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(top_spam.index[::-1], top_spam.values[::-1], color='#e85d20', edgecolor='white')
axes[0].set_title('Top 15 Spam Words')
axes[0].set_xlabel('Log-ratio (spam/ham)')

axes[1].barh(top_ham.index, top_ham.abs().values, color='#1e5fa8', edgecolor='white')
axes[1].set_title('Top 15 Ham Words')
axes[1].set_xlabel('Log-ratio (ham/spam)')

plt.suptitle('Most Discriminative Words — Naive Bayes Spam Classifier', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 The model learned these patterns purely from labeled examples — no rules were written by hand")

In [ ]:
answers = {
    "q1": "",  # What is the conditional independence assumption in Naive Bayes?
    "q2": "",  # What type of Naive Bayes would you use for word counts in documents?
    "q3": "",  # What is Laplace smoothing and why is it needed?
    "q4": "",  # Why does Naive Bayes sometimes outperform logistic regression on small datasets?
    "q5": "",  # What is one real-world scenario where the independence assumption is clearly violated but NB still works?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*